In [ ]:
import pandas as pd
import numpy as np
import re
import faiss
from rank_bm25 import BM25Okapi
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer

books = pd.read_parquet('D:/documents/Smart_Library/data/processed/books_with_mood.parquet')
embeddings = np.load('D:/documents/Smart_Library/data/processed/book_embedding.npy')
ratings = pd.read_parquet('D:/documents/Smart_Library/data/processed/ratings_clean.parquet')

print(len(books) ==  embeddings.shape[0]) 

In [ ]:
def build_search_text(row):
    parts = [str(row['title'])]
    if pd.notna(row['author']):
        parts.append(str(row['author']))
    if pd.notna(row['categories']):
        parts.append(str(row['categories']))
    return ' '.join(parts).lower()

books['search_text'] = books.apply(build_search_text, axis=1)
tokenized_corpus = [text.split() for text  in books['search_text']]
bm25 = BM25Okapi(tokenized_corpus)

scores = bm25.get_scores("harry potter".split())
top5 = np.argsort(scores)[-5:][::-1]
print(books.iloc[top5][['title','author']])

In [ ]:
print(books.columns.tolist())

In [ ]:
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings.astype('float32'))
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

query_vector = model_embed.encode(["a story about loniliness and isolation"]).astype('float32')
distances, indices =  faiss_index.search(query_vector,k = 5)
print(books.iloc[indices[0]][['title', 'description']])

In [ ]:
import inspect
from search import load_search_system
print(inspect.getsource(load_search_system))

In [ ]:
import os
print(os.listdir('D:/documents/Smart_Library/data/processed'))

In [ ]:
import search
print(search.__file__)

import inspect
print(inspect.getsource(search.live_lookup))

In [1]:
import sys
sys.path.append('D:/documents/Smart_Library/src')
from search import live_lookup

results = live_lookup("The Midnight Library Matt Haig")
for r in results[:3]:
    print(r['title'], '-', r['author'])

d:\documents\Smart_Library\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The Midnight Library - Matt Haig
The Comfort Book / Reasons To Stay Alive / The Midnight Library - Matt Haig
Life Impossible - Matt Haig


In [2]:
import pandas as pd

for filename in ['books_clean.parquet', 'books_with_mood.parquet', 'books_search_ready.parquet']:
    path = f'D:/documents/Smart_Library/data/processed/{filename}'
    df = pd.read_parquet(path)

    mask = df['source'] == 'book_crossing_only'
    df.loc[mask, 'cover_url'] = (
        'https://covers.openlibrary.org/b/isbn/' + df.loc[mask, 'isbn13'].astype(str) + '-L.jpg'
    )

    df.to_parquet(path, index=False)
    print(f"Fixed and saved: {filename}")

Fixed and saved: books_clean.parquet
Fixed and saved: books_with_mood.parquet
Fixed and saved: books_search_ready.parquet


In [3]:
import pandas as pd
import ftfy

files_to_fix = ['books_clean.parquet', 'books_with_mood.parquet', 'books_search_ready.parquet']
text_columns = ['title', 'author', 'description']

for filename in files_to_fix:
    path = f'D:/documents/Smart_Library/data/processed/{filename}'
    df = pd.read_parquet(path)

    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: ftfy.fix_text(x) if pd.notna(x) else x)

    # books_search_ready.parquet also needs search_text rebuilt, since it was built from the corrupted title
    if 'search_text' in df.columns:
        df['search_text'] = (
            df['title'].fillna('') + ' ' + df['author'].fillna('') + ' ' + df['categories'].fillna('')
        ).str.lower()

    df.to_parquet(path, index=False)
    print(f"Fixed and saved: {filename}")

# Quick verification
df_check = pd.read_parquet('D:/documents/Smart_Library/data/processed/books_search_ready.parquet')
print(df_check[df_check['title'].str.contains('nix', case=False, na=False)]['title'].head(5))

Fixed and saved: books_clean.parquet
Fixed and saved: books_with_mood.parquet
Fixed and saved: books_search_ready.parquet
58      Harry Potter and the Order of the Phoenix (Boo...
3243                                          Red Phoenix
3731                                      Spandau Phoenix
4656                                       Rising Phoenix
7390                               Naked Came the Phoenix
Name: title, dtype: str
